# 07 - 消融实验分析

## 本 Notebook 的目标

1. **数据组成分析**：展示 6 组消融实验的数据构成差异
2. **可视化对比**：饼图展示每组消融去掉了什么
3. **Tulu 3 论文预测**：基于论文结论，预测各消融的影响
4. **安全正交性假说**：为什么安全数据不会损害通用能力

### 消融实验设计（对应 Tulu 3 论文 Section 3.1）

| 实验名 | 去掉的数据 | 验证假说 |
|--------|-----------|----------|
| full | 无（对照基准） | 完整训练效果 |
| no_safety | WildGuardMix + WildJailbreak | 安全数据的贡献 |
| no_math | NuminaMath + Persona Math | 数学数据的贡献 |
| no_coconot | CoCoNot | 防过度拒绝数据的贡献 |
| no_norobots | No Robots | 人工高质量数据的价值 |
| no_flan | FLAN v2 | 通用指令数据的贡献 |

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['Arial Unicode MS', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## A. 消融数据集构成分析

每组消融实验从完整 SFT 数据中排除特定子集，观察对模型能力和安全性的影响。

In [ ]:
# 加载所有消融数据集的 source 分布
ablation_names = ["full", "no_safety", "no_math", "no_coconot", "no_norobots", "no_flan"]
ablation_descriptions = {
    "full": "完整数据（对照基准）",
    "no_safety": "去掉安全数据",
    "no_math": "去掉数学数据",
    "no_coconot": "去掉 CoCoNot",
    "no_norobots": "去掉 No Robots",
    "no_flan": "去掉 FLAN v2",
}

ablation_data = {}
for name in ablation_names:
    path = PROJECT_ROOT / f"data/sft_mix/ablation_{name}.jsonl"
    sources = Counter()
    total = 0
    with open(path) as f:
        for line in f:
            if line.strip():
                sample = json.loads(line)
                source = sample.get("source", "unknown")
                source_short = source.split("/")[-1] if "/" in source else source
                sources[source_short] += 1
                total += 1
    ablation_data[name] = {"sources": dict(sources), "total": total}

# 打印汇总表
print(f"{'实验':<15} {'样本数':<10} {'减少量':<10} {'减少比例':<10} {'说明'}")
print("-" * 75)
full_total = ablation_data["full"]["total"]
for name in ablation_names:
    total = ablation_data[name]["total"]
    diff = full_total - total
    pct = diff / full_total * 100 if full_total > 0 else 0
    desc = ablation_descriptions[name]
    print(f"{name:<15} {total:<10} {diff:<10} {pct:<10.1f}% {desc}")

In [ ]:
# 饼图: 完整数据构成 vs 各消融
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
colors_palette = plt.cm.Set3(np.linspace(0, 1, 10))

for idx, name in enumerate(ablation_names):
    ax = axes[idx // 3, idx % 3]
    sources = ablation_data[name]["sources"]
    total = ablation_data[name]["total"]
    
    labels = list(sources.keys())
    sizes = list(sources.values())
    
    # 小于 3% 的合并为 "other"
    threshold = total * 0.03
    main_labels, main_sizes, other_size = [], [], 0
    for l, s in zip(labels, sizes):
        if s >= threshold:
            main_labels.append(l)
            main_sizes.append(s)
        else:
            other_size += s
    if other_size > 0:
        main_labels.append("other")
        main_sizes.append(other_size)
    
    wedges, texts, autotexts = ax.pie(
        main_sizes, labels=main_labels, autopct='%1.0f%%',
        colors=colors_palette[:len(main_sizes)], textprops={'fontsize': 8}
    )
    for autotext in autotexts:
        autotext.set_fontsize(7)
    
    diff = full_total - total
    ax.set_title(f"{name}\n({total} samples, -{diff})", fontweight='bold', fontsize=11)

plt.suptitle("消融实验: 数据构成对比", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'results/figures/ablation_data_composition.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved to results/figures/ablation_data_composition.png")

In [ ]:
# 条形图: 各消融的样本数量对比
fig, ax = plt.subplots(figsize=(10, 6))

names = list(ablation_names)
totals = [ablation_data[n]["total"] for n in names]
bar_colors = ["#2ecc71" if n == "full" else "#e74c3c" for n in names]

bars = ax.barh(names, totals, color=bar_colors, height=0.5, edgecolor='white')

for bar, total in zip(bars, totals):
    diff = full_total - total
    label = f"{total} (-{diff})" if diff > 0 else f"{total} (baseline)"
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=10)

ax.set_xlabel("样本数量")
ax.set_title("各消融实验的训练数据量", fontweight='bold', fontsize=13)
ax.set_xlim(0, max(totals) * 1.25)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("\n关键观察:")
print(f"- no_safety 减少最多 (-{full_total - ablation_data['no_safety']['total']} 样本, 安全数据占比最大)")
print(f"- no_math 减少最少之一 (-{full_total - ablation_data['no_math']['total']} 样本, 数学数据相对较少)")

## B. Tulu 3 论文预测

基于 Tulu 3 论文（Ivison et al., 2024）的实验结论，我们可以预测各消融的影响：

### 预测表

| 消融实验 | 预测: Benchmark 影响 | 预测: 安全影响 | 论文依据 |
|---------|---------------------|---------------|----------|
| no_safety | 无显著影响 | ASR 显著上升 | **安全正交性**: 安全数据不影响通用能力 |
| no_math | 数学 benchmark 下降 | 无显著影响 | 数学能力需要专门数据 |
| no_coconot | 无显著影响 | Over-refusal 上升 | CoCoNot 专门训练"不要过度拒绝" |
| no_norobots | 对话质量下降 | 无显著影响 | 人工数据提供高质量对话模板 |
| no_flan | 通用能力轻微下降 | 无显著影响 | FLAN v2 覆盖面广但单项贡献小 |

In [ ]:
# 可视化: 预测影响矩阵
fig, ax = plt.subplots(figsize=(10, 6))

# 影响矩阵 (行: 消融, 列: 能力维度)
# 值: -2=严重下降, -1=轻微下降, 0=无影响, 1=轻微上升
ablation_labels = ["no_safety", "no_math", "no_coconot", "no_norobots", "no_flan"]
dimension_labels = ["HellaSwag", "Math", "ASR \u2191", "Over-refusal \u2191", "Dialog Quality"]

# 预测矩阵 (基于 Tulu 3 论文)
impact_matrix = np.array([
    [0,  0, -2,  0,  0],  # no_safety: ASR 严重上升
    [0, -2,  0,  0,  0],  # no_math: 数学下降
    [0,  0,  0, -1,  0],  # no_coconot: Over-refusal 上升
    [0,  0,  0,  0, -1],  # no_norobots: 对话质量下降
    [-1, 0,  0,  0,  0],  # no_flan: 通用能力轻微下降
])

cmap = plt.cm.RdYlGn
im = ax.imshow(impact_matrix, cmap=cmap, aspect='auto', vmin=-2, vmax=1)

ax.set_xticks(range(len(dimension_labels)))
ax.set_xticklabels(dimension_labels, fontsize=10)
ax.set_yticks(range(len(ablation_labels)))
ax.set_yticklabels(ablation_labels, fontsize=10)

# 标注
impact_text = {-2: "\u2193\u2193", -1: "\u2193", 0: "\u2014", 1: "\u2191"}
for i in range(len(ablation_labels)):
    for j in range(len(dimension_labels)):
        text = impact_text[impact_matrix[i, j]]
        color = "white" if abs(impact_matrix[i, j]) >= 2 else "black"
        ax.text(j, i, text, ha='center', va='center', fontsize=14, color=color, fontweight='bold')

ax.set_title("消融实验预测影响矩阵\n(基于 Tulu 3 论文)", fontweight='bold', fontsize=13)
plt.colorbar(im, ax=ax, label="Impact", shrink=0.8)
plt.tight_layout()
plt.show()

print("\u2193\u2193 = 严重下降, \u2193 = 轻微下降, \u2014 = 无影响, \u2191 = 上升")

## C. 安全正交性假说

Tulu 3 论文中的一个核心发现是 **安全正交性（Safety Orthogonality）**：

> 安全训练数据对模型通用能力的影响可以忽略不计。这意味着可以独立优化安全性，而不必担心通用能力下降。

### 理论解释

1. **参数空间分离**: 安全拒绝行为和通用推理能力占据模型参数空间的不同子空间
2. **梯度方向正交**: 安全数据产生的梯度更新与通用能力数据的梯度近似正交
3. **实验验证**: no_safety 消融中，通用 benchmark 分数不受影响，但安全指标显著恶化

In [ ]:
# 可视化: 安全正交性概念图
fig, ax = plt.subplots(figsize=(8, 8))

# 绘制概念向量
ax.arrow(0, 0, 0.8, 0, head_width=0.03, head_length=0.03, fc='#3498db', ec='#3498db', linewidth=2)
ax.arrow(0, 0, 0, 0.8, head_width=0.03, head_length=0.03, fc='#e74c3c', ec='#e74c3c', linewidth=2)

# 混合训练（两者兼顾）
ax.arrow(0, 0, 0.6, 0.6, head_width=0.03, head_length=0.03, fc='#2ecc71', ec='#2ecc71', linewidth=2)

ax.text(0.85, -0.05, "通用能力\n(HellaSwag, MMLU)", fontsize=11, color='#3498db', fontweight='bold')
ax.text(-0.15, 0.85, "安全能力\n(ASR\u2193, Refusal)", fontsize=11, color='#e74c3c', fontweight='bold')
ax.text(0.65, 0.65, "Tulu 3\n混合训练", fontsize=11, color='#2ecc71', fontweight='bold')

# 正交标记
ax.plot([0.05, 0.05, 0], [0, 0.05, 0.05], 'k-', linewidth=1)

ax.set_xlim(-0.2, 1.1)
ax.set_ylim(-0.15, 1.1)
ax.set_aspect('equal')
ax.set_title("安全正交性假说示意图\n(Safety Orthogonality Hypothesis)", fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print("正交性含义: 安全训练不干扰通用能力的梯度方向")
print("实际影响: 可以放心地增加安全数据而不牺牲模型性能")

## D. 运行完整消融实验

消融数据已经准备好，完整训练+评估需要约 3 小时。如需运行：

```bash
# 在项目根目录下运行
PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 python scripts/run_ablation.py
```

### 预期输出

```
results/eval_results/ablation_full.json
results/eval_results/ablation_no_safety.json
results/eval_results/ablation_no_math.json
results/eval_results/ablation_no_coconot.json
results/eval_results/ablation_no_norobots.json
results/eval_results/ablation_no_flan.json
results/eval_results/ablation_summary.json
```

每个 JSON 文件包含 benchmark 分数、安全指标和最终训练 loss。

### 消融脚本说明

`scripts/run_ablation.py` 对每组消融执行：
1. 从完整 SFT 数据中排除指定子集
2. 用过滤后的数据训练 SFT 模型
3. 运行 benchmark 评估
4. 运行安全评估
5. 保存结果到 JSON

In [ ]:
# 检查是否有已完成的消融结果
import glob

ablation_results_files = glob.glob(str(PROJECT_ROOT / "results/eval_results/ablation_*.json"))

if ablation_results_files:
    print("找到消融实验结果:")
    for f in sorted(ablation_results_files):
        print(f"  {Path(f).name}")
    
    # 如果有汇总文件，展示结果
    summary_file = PROJECT_ROOT / "results/eval_results/ablation_summary.json"
    if summary_file.exists():
        with open(summary_file) as f:
            abl_summary = json.load(f)
        
        print("\n" + "=" * 80)
        print(f"{'实验':<15} {'HellaSwag':<12} {'ASR↓':<10} {'Over-ref↓':<12} {'说明'}")
        print("-" * 80)
        for name, result in abl_summary.items():
            hs = result.get("benchmarks", {}).get("hellaswag", "—")
            asr = result.get("safety", {}).get("ASR", "—")
            ore = result.get("safety", {}).get("Over-refusal", "—")
            desc = result.get("description", "")
            hs_str = f"{hs:.4f}" if isinstance(hs, float) else str(hs)
            asr_str = f"{asr:.1f}%" if isinstance(asr, (int, float)) else str(asr)
            or_str = f"{ore:.1f}%" if isinstance(ore, (int, float)) else str(ore)
            print(f"{name:<15} {hs_str:<12} {asr_str:<10} {or_str:<12} {desc}")
        print("=" * 80)
else:
    print("消融实验尚未运行。")
    print("消融数据已准备好，运行以下命令开始完整实验:")
    print("  PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 python scripts/run_ablation.py")
    print(f"\n已准备的消融数据集:")
    for name in ablation_names:
        path = PROJECT_ROOT / f"data/sft_mix/ablation_{name}.jsonl"
        if path.exists():
            total = ablation_data[name]["total"]
            print(f"  {name}: {total} samples")

## E. 总结与建议

### 消融实验的核心价值

1. **量化每个组件的贡献**: 不是所有数据都同等重要
2. **验证安全正交性**: 安全数据可以独立优化
3. **指导数据配比**: 知道哪些数据对哪些能力有贡献后，可以更精准地调整比例

### 基于 Tulu 3 的最佳实践

- 安全数据（WildGuard + WildJailbreak）是降低 ASR 的关键
- CoCoNot 是控制 Over-refusal 的关键
- 数学数据对通用 benchmark 影响较小，但对数学特定任务贡献大
- FLAN v2 提供广泛覆盖但每项贡献较小，属于"长尾"数据

### 下一步

→ **Notebook 08**: 三方对比 Dashboard（综合可视化）